# 03 - Tokenizer comparison

Presentation-only notebook. Generate or refresh results with `python -m yelptext analyze tokenizers --config configs/full.toml`. Sampling, tokenization, counting, benchmarking, qualitative cases, and artifact export run in the registered command, not here.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

def find_root():
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / 'pyproject.toml').is_file():
            return path
    raise RuntimeError('Run inside the project repository')

ROOT = find_root()
ARTIFACTS = ROOT / 'artifacts' / 'tokenizer_comparison'
metadata = json.loads((ARTIFACTS / 'summary.json').read_text(encoding='utf-8'))
summary = pd.read_csv(ARTIFACTS / 'summary.csv')
benchmark = pd.read_csv(ARTIFACTS / 'runtime_benchmark.csv')
qualitative = pd.read_csv(ARTIFACTS / 'qualitative_cases.csv')
if metadata.get('sample_size') != 50_000 or len(summary) != 3:
    raise RuntimeError('Run: python -m yelptext analyze tokenizers --config configs/full.toml')


## Controlled sample comparison

In [ ]:
display(summary.style.format({
    'tokens': '{:,.0f}', 'vocabulary': '{:,.0f}',
    'hapax_share': '{:.2%}', 'processing_seconds': '{:.2f}',
    'reviews_per_second_including_counting': '{:,.0f}',
}))
display(benchmark.style.format({
    'median_seconds': '{:.4f}', 'median_reviews_per_second': '{:,.0f}',
}))
display(Image(filename=str(ARTIFACTS / 'tokenizer_comparison.png')))


## Qualitative policy cases

These constructed examples illustrate boundary policy; they are not annotated-corpus accuracy measurements.

In [ ]:
pd.set_option('display.max_colwidth', 180)
qualitative


## Interpretation

All methods receive the shared tokenizer's NFC, apostrophe, width, and case normalization. No stopword removal, stemming, lemmatization, or punctuation filtering is applied. Vocabulary size and speed are descriptive rather than accuracy measures. The qualitative cases explain how the custom policy preserves contractions, structured numbers, URLs, hashtags, emoticons, and composed emoji while splitting run-on hyphenated words. Runtime values are machine-dependent and describe an offline preprocessing trade-off.